# 01 — Exploratory Data Analysis
**Credit Risk Vintage Curves & Survival Analysis — Lending Club (2007-2018)**

This notebook performs the initial exploration of 2.2M+ loans to understand portfolio composition, default distributions, and temporal trends.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'

In [ ]:
df = pd.read_csv('../data/clean_loans.csv', parse_dates=['issue_date', 'last_pymnt_date'])
print(f"Dataset: {len(df):,} loans")
print(f"Date range: {df['issue_date'].min().date()} → {df['issue_date'].max().date()}")
print(f"Overall default rate: {df['is_default'].mean():.2%}")
df.head()

## 1. Portfolio KPIs

In [ ]:
print('='*60)
print('PORTFOLIO SUMMARY')
print('='*60)
print(f"Total Loans:          {len(df):>15,}")
print(f"Total Funded:         ${df['funded_amnt'].sum():>14,.0f}")
print(f"Avg Loan Amount:      ${df['loan_amnt'].mean():>14,.0f}")
print(f"Avg FICO Score:       {df['fico_score'].mean():>15.0f}")
print(f"Avg Interest Rate:    {df['int_rate'].mean():>14.2f}%")
print(f"Overall Default Rate: {df['is_default'].mean():>14.2%}")
print(f"Total Defaults:       {df['is_default'].sum():>15,}")
print(f"Total Loss Amount:    ${df['loss_amount'].sum():>14,.0f}")

## 2. Default Rate by Grade

In [ ]:
grade_stats = df.groupby('grade').agg(
    loan_count=('id', 'count'),
    default_rate=('is_default', 'mean'),
    avg_interest_rate=('int_rate', 'mean'),
    avg_fico=('fico_score', 'mean'),
    avg_loan_amount=('loan_amnt', 'mean'),
    total_funded=('funded_amnt', 'sum'),
    total_loss=('loss_amount', 'sum'),
).round(4)

grade_stats['loss_rate'] = (grade_stats['total_loss'] / grade_stats['total_funded']).round(4)
print(grade_stats.to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Default rate by grade
colors = ['#27ae60', '#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#c0392b', '#8e44ad']
axes[0].bar(grade_stats.index, grade_stats['default_rate'] * 100, color=colors)
axes[0].set_title('Default Rate by Loan Grade', fontweight='bold')
axes[0].set_ylabel('Default Rate (%)')
axes[0].set_xlabel('Grade')

# Interest rate by grade
axes[1].bar(grade_stats.index, grade_stats['avg_interest_rate'], color=colors)
axes[1].set_title('Avg Interest Rate by Grade', fontweight='bold')
axes[1].set_ylabel('Interest Rate (%)')
axes[1].set_xlabel('Grade')

# Loan volume by grade
axes[2].bar(grade_stats.index, grade_stats['loan_count'] / 1000, color=colors)
axes[2].set_title('Loan Volume by Grade (thousands)', fontweight='bold')
axes[2].set_ylabel('Loans (K)')
axes[2].set_xlabel('Grade')

plt.tight_layout()
plt.savefig('../outputs/default_rate_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Quarterly Default Rate Trend

In [ ]:
quarterly = df.groupby('issue_quarter').agg(
    loan_count=('id', 'count'),
    default_rate=('is_default', 'mean'),
    avg_fico=('fico_score', 'mean'),
    avg_int_rate=('int_rate', 'mean'),
).reset_index()

fig, ax1 = plt.subplots(figsize=(16, 6))
ax2 = ax1.twinx()

ax1.bar(range(len(quarterly)), quarterly['loan_count'] / 1000, alpha=0.3, color='#3498db', label='Loan Volume (K)')
ax2.plot(range(len(quarterly)), quarterly['default_rate'] * 100, color='#e74c3c', linewidth=2.5, marker='o', markersize=3, label='Default Rate %')

ax1.set_xlabel('Origination Quarter')
ax1.set_ylabel('Loan Volume (thousands)', color='#3498db')
ax2.set_ylabel('Default Rate (%)', color='#e74c3c')
ax1.set_title('Quarterly Loan Volume & Default Rate', fontweight='bold', fontsize=14)

# Show every 4th label
ax1.set_xticks(range(0, len(quarterly), 4))
ax1.set_xticklabels(quarterly['issue_quarter'].iloc[::4], rotation=45, ha='right')

fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.95))
plt.tight_layout()
plt.savefig('../outputs/quarterly_default_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Default Rate by Loan Purpose

In [ ]:
purpose_stats = df.groupby('purpose').agg(
    loan_count=('id', 'count'),
    default_rate=('is_default', 'mean'),
).sort_values('default_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(purpose_stats)))
ax.barh(purpose_stats.index, purpose_stats['default_rate'] * 100, color=colors)
ax.set_xlabel('Default Rate (%)')
ax.set_title('Default Rate by Loan Purpose', fontweight='bold')

for i, (val, cnt) in enumerate(zip(purpose_stats['default_rate'], purpose_stats['loan_count'])):
    ax.text(val * 100 + 0.2, i, f'{val:.1%} ({cnt:,})', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/default_rate_by_purpose.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. FICO Score Distribution by Default Status

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
df[df['is_default']==0]['fico_score'].hist(bins=50, alpha=0.6, label='Non-Default', color='#2ecc71', ax=ax, density=True)
df[df['is_default']==1]['fico_score'].hist(bins=50, alpha=0.6, label='Default', color='#e74c3c', ax=ax, density=True)
ax.set_title('FICO Score Distribution — Defaults vs Non-Defaults', fontweight='bold')
ax.set_xlabel('FICO Score')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/fico_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# FICO bucket summary
fico_summary = df.groupby('fico_bucket', observed=True).agg(
    loan_count=('id', 'count'),
    default_rate=('is_default', 'mean'),
    avg_int_rate=('int_rate', 'mean'),
).round(4)
print(fico_summary)

## 6. State-Level Analysis

In [ ]:
state_stats = df.groupby('addr_state').agg(
    loan_count=('id', 'count'),
    default_rate=('is_default', 'mean'),
    total_funded=('funded_amnt', 'sum'),
).sort_values('loan_count', ascending=False)

print('Top 10 States by Volume:')
print(state_stats.head(10).to_string())
print(f'\nHighest default rate: {state_stats["default_rate"].idxmax()} ({state_stats["default_rate"].max():.2%})')
print(f'Lowest default rate:  {state_stats["default_rate"].idxmin()} ({state_stats["default_rate"].min():.2%})')